In [ ]:
import pandas as pd
import numpy as np
import joblib
from collections import deque
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

print("=== SISTEMA DE INFERENCIA COLDTRACK - VERSIÓN ESTABLE FINAL ===\n")

# Cargar modelo, columnas y umbral
model = joblib.load("coldtrack_rf_model_final.pkl")
feature_columns = joblib.load("feature_columns_final.pkl")
threshold = joblib.load("decision_threshold.pkl")

print(f"Modelo cargado | Umbral: {threshold}\n")

# Buffer histórico
BUFFER_SIZE = 360
lecturas_buffer = deque(maxlen=BUFFER_SIZE)

def predecir_mantenimiento(nueva_lectura: dict):
    lecturas_buffer.append(nueva_lectura)
    
    if len(lecturas_buffer) < 10:
        return {
            "probabilidad": 0.0,
            "requiere_mantenimiento": False,
            "umbral": threshold,
            "confianza": "Insuficiente datos",
            "buffer_size": len(lecturas_buffer)
        }
    
    # Crear DataFrame
    df = pd.DataFrame(lecturas_buffer)
    
    # Convertir Timestamp de forma segura
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format="%A %d.%m.%Y -- %H:%M:%S", errors='coerce')
    
    df = df.sort_values('Timestamp').reset_index(drop=True)
    
    # Feature Engineering
    df['temp_diff_setpoint'] = df['inTemp'] - 2.5
    df['hour'] = df['Timestamp'].dt.hour
    df['day_of_week'] = df['Timestamp'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    df['DoorOPEN_flag'] = (df['DoorStatus'] == 'DoorOPEN').astype(int)
    
    n = len(df)
    
    # Rolling features con protección
    df['inTemp_ma_5min']   = df['inTemp'].rolling(window=min(60, n), min_periods=1).mean()
    df['inTemp_ma_15min']  = df['inTemp'].rolling(window=min(180, n), min_periods=1).mean()
    df['inTemp_ma_30min']  = df['inTemp'].rolling(window=min(360, n), min_periods=1).mean()
    
    df['Vibration_ma_5min'] = df['Vibration'].rolling(window=min(60, n), min_periods=1).mean()
    df['Vibration_ma_15min'] = df['Vibration'].rolling(window=min(180, n), min_periods=1).mean()
    
    df['ConsumoElectrico_ma_5min'] = df['ConsumoElectrico'].rolling(window=min(60, n), min_periods=1).mean()
    
    df['door_open_last_hour'] = df['DoorOPEN_flag'].rolling(window=min(720, n), min_periods=1).sum()
    
    # Tendencia segura
    def safe_slope(x):
        if len(x) < 2:
            return 0.0
        return np.polyfit(range(len(x)), x, 1)[0]
    
    window = min(360, n)
    if window >= 10:
        df['inTemp_trend_30min'] = df['inTemp'].rolling(window=window, min_periods=10).apply(safe_slope, raw=True)
    else:
        df['inTemp_trend_30min'] = 0.0
    
    # Última lectura
    df_last = df.iloc[[-1]]
    X_new = df_last[feature_columns]
    
    # Predicción
    proba = model.predict_proba(X_new)[0, 1]
    requiere = proba >= threshold
    
    confianza = "Alta" if proba > 0.85 else "Media" if proba > 0.65 else "Baja"
    
    return {
        "probabilidad": round(proba, 4),
        "requiere_mantenimiento": bool(requiere),
        "umbral": threshold,
        "confianza": confianza,
        "buffer_size": len(lecturas_buffer)
    }


# ====================== PRUEBAS ======================
print("1. Serie de lecturas NORMALES (2 minutos):")
base_time = datetime(2025, 1, 18, 14, 35, 0)

for i in range(25):
    lectura = {
        "inTemp": round(2.9 + np.random.uniform(-0.4, 0.4), 2),
        "InHumid": round(63.0 + np.random.uniform(-4, 4), 1),
        "outTemp": 23.5,
        "outHumid": 71.0,
        "DoorStatus": "DoorCLOSED",
        "PowerStatus": "PowerON",
        "ConsumoElectrico": round(102.0 + np.random.uniform(-8, 8), 1),
        "CompressorStatus": "CompressorON",
        "Vibration": round(0.20 + np.random.uniform(-0.05, 0.08), 2),
        "Timestamp": (base_time + timedelta(seconds=i*5)).strftime("%A %d.%m.%Y -- %H:%M:%S")
    }
    res = predecir_mantenimiento(lectura)
    if i % 5 == 0 or i == 24:
        print(f"  Lectura {i+1:2d} | Prob: {res['probabilidad']:.4f} | Mant: {res['requiere_mantenimiento']} | Buffer: {res['buffer_size']}")

print("\n2. Transición a ANOMALÍA (temperatura subiendo + puerta abierta):")
for i in range(15):
    lectura = {
        "inTemp": round(4.0 + i*0.28, 2),           # temperatura subiendo
        "InHumid": 81.0,
        "outTemp": 26.5,
        "outHumid": 76.0,
        "DoorStatus": "DoorOPEN" if i > 3 else "DoorCLOSED",
        "PowerStatus": "PowerON",
        "ConsumoElectrico": round(155.0 + i*6, 1),
        "CompressorStatus": "CompressorON",
        "Vibration": round(0.65 + i*0.04, 2),
        "Timestamp": (base_time + timedelta(minutes=3, seconds=i*5)).strftime("%A %d.%m.%Y -- %H:%M:%S")
    }
    res = predecir_mantenimiento(lectura)
    print(f"  Lectura {i+1:2d} | Prob: {res['probabilidad']:.4f} | Mant: {res['requiere_mantenimiento']} | Confianza: {res['confianza']}")

print("\n¡Pruebas finalizadas!")

=== SISTEMA DE INFERENCIA COLDTRACK - VERSIÓN ESTABLE FINAL ===

Modelo cargado | Umbral: 0.68

1. Serie de lecturas NORMALES (2 minutos):
  Lectura  1 | Prob: 0.0000 | Mant: False | Buffer: 1
  Lectura  6 | Prob: 0.0000 | Mant: False | Buffer: 6
  Lectura 11 | Prob: 0.0000 | Mant: False | Buffer: 11
  Lectura 16 | Prob: 0.0000 | Mant: False | Buffer: 16
  Lectura 21 | Prob: 0.0000 | Mant: False | Buffer: 21
  Lectura 25 | Prob: 0.0000 | Mant: False | Buffer: 25

2. Transición a ANOMALÍA (temperatura subiendo + puerta abierta):
  Lectura  1 | Prob: 0.0038 | Mant: False | Confianza: Baja
  Lectura  2 | Prob: 0.0038 | Mant: False | Confianza: Baja
  Lectura  3 | Prob: 0.0038 | Mant: False | Confianza: Baja
  Lectura  4 | Prob: 0.0022 | Mant: False | Confianza: Baja
  Lectura  5 | Prob: 0.0082 | Mant: False | Confianza: Baja
  Lectura  6 | Prob: 0.0082 | Mant: False | Confianza: Baja
  Lectura  7 | Prob: 0.0111 | Mant: False | Confianza: Baja
  Lectura  8 | Prob: 0.0112 | Mant: False | Co